In [ ]:
import os
import math
import json
import numpy as np
import pandas as pd
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_PATH = "Nitrate_Benchmark_4.csv"

OUT_DIR = "pinn_uq_outputs"
MODEL_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

ENSEMBLE_SIZE = 50
SEEDS = None   
BOOTSTRAP_TRAIN_SIMS = False      

CAL_FRAC = 0.10                   
ALPHAS = [0.10, 0.05]
Z_EPI = 1.0                       
BETA = 0.0                       

BATCH_SIZE = 65536
EPOCHS_ADAM = 2000
LR_ADAM = 1e-3
WD_ADAMW = 1e-4
LBFGS_MAX_ITER = 2000
USE_COLLOCATION = True
COLLOC_MULTIPLIER = 0.5

lambda_phys_base  = 0.0
lambda_phys_max   = 1.0
lambda_redox_base = 0.0
lambda_redox_max  = 0.2
lambda_nonneg     = 0.1
warmup_epochs     = 400

lambda_bc_in  = 1.0
lambda_bc_out = 1.0

        low=0,
        high=2**31 - 1,
        size=ENSEMBLE_SIZE,
        dtype=np.int64
    ).tolist()

input_columns = [
    'dist_x', 'time', 'C_DOC', 'porosity', 'velocity', 'dispersivity',
    'C_NO3', 'k_Fe', 'k_C', 'C_Fe3', 'K_Fe'
]
output_column = 'N(5)(mol/kgw)'
CANDIDATE_CONST_COLS = [
    'porosity', 'velocity', 'dispersivity', 'k_Fe', 'k_C', 'C_Fe3', 'K_Fe'
]


class SurrogatePINN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 64),  nn.Tanh(),
            nn.Linear(64, 64),   nn.Tanh(),
            nn.Linear(64, 32),   nn.Tanh(),
            nn.Linear(32, 1)
        )
    def forward(self, x):    SEED_LIST = rng_seeds.integers(
        low=0,
        high=2**31 - 1,
        size=ENSEMBLE_SIZE,
        dtype=np.int64
    ).tolist()
        return self.net(x)


def compute_nonnegativity_penalty(C_phys):
    c = np.asarray(C_phys).reshape(-1)
    neg = c < 0
    return float(np.mean(c[neg]**2)) if np.any(neg) else 0.0

def compute_redox_sequence_penalty(df_phys, window=3, pe_min=-300.0, pe_max=0.0, eps_NO3=1e-6, eps_Fe3=1e-9):
    if 'N(5)(mol/kgw)_predicted' not in df_phys.columns or 'C_Fe3' not in df_phys.columns:
        return 0.0
    Df = df_phys.sort_values(['dist_x', 'time']).reset_index(drop=True)
    pe_v = Df['pe'].values.astype(float) if 'pe' in Df.columns else np.zeros(len(Df))
    NO3p = Df['N(5)(mol/kgw)_predicted'].values.astype(float)
    Fe3  = Df['C_Fe3'].values.astype(float)
    NO3_s = pd.Series(NO3p).rolling(window=window, min_periods=1, center=True).mean().values
    Fe3_s = pd.Series(Fe3).rolling(window=window, min_periods=1, center=True).mean().values
    dFe3  = np.diff(Fe3_s, prepend=Fe3_s[0])
    pe_w = np.zeros_like(pe_v)
    pe_w[pe_v <= pe_min] = 1.0
    mid = (pe_v > pe_min) & (pe_v < pe_max)
    if np.any(mid):
        pe_w[mid] = (pe_max - pe_v[mid]) / (pe_max - pe_min)
    mask = (NO3_s > eps_NO3) & (Fe3_s > eps_Fe3) & (dFe3 < 0)
    if not np.any(mask): return 0.0
    Fe3_mean = max(np.mean(Fe3_s), eps_Fe3)
    rel_drop = np.minimum((-dFe3[mask]) / Fe3_mean, 2.0)
    w = pe_w[mask]
    if rel_drop.size == 0: return 0.0
    return float(np.mean(rel_drop * w))

ix_x       = input_columns.index('dist_x')
ix_t       = input_columns.index('time')
ix_DOC     = input_columns.index('C_DOC')
ix_phi     = input_columns.index('porosity')
ix_v       = input_columns.index('velocity')
ix_alphaL  = input_columns.index('dispersivity')
ix_kFe     = input_columns.index('k_Fe')
ix_kC      = input_columns.index('k_C')
ix_Fe3     = input_columns.index('C_Fe3')
ix_KFe     = input_columns.index('K_Fe')

def pde_residual_mean_abs(C_scaled, xb_scaled, mu_X_t, std_X_t, mu_y_t, std_y_t, device):
    if not xb_scaled.requires_grad:
        xb_scaled.requires_grad_(True)
    C = C_scaled * std_y_t + mu_y_t
    std_x = std_X_t[ix_x].view(1, 1)
    std_t = std_X_t[ix_t].view(1, 1)
    ones = torch.ones_like(C)
    grad_C = torch.autograd.grad(C, xb_scaled, grad_outputs=ones,
                                 create_graph=True, retain_graph=True)[0]
    dCdx_s = grad_C[:, ix_x:ix_x+1]
    dCdt_s = grad_C[:, ix_t:ix_t+1]
    dCdx = dCdx_s / std_x
    dCdt = dCdt_s / std_t
    grad_dCdx_s = torch.autograd.grad(dCdx_s, xb_scaled,
                                      grad_outputs=torch.ones_like(dCdx_s),
                                      create_graph=True, retain_graph=True)[0]
    d2Cdx2_s = grad_dCdx_s[:, ix_x:ix_x+1]
    d2Cdx2   = d2Cdx2_s / (std_x * std_x)
    xb_phys = xb_scaled * std_X_t + mu_X_t
    phi     = xb_phys[:, ix_phi:ix_phi+1]
    v       = xb_phys[:, ix_v:ix_v+1]
    alphaL  = xb_phys[:, ix_alphaL:ix_alphaL+1]
    DL      = alphaL * v
    C_NO3 = C.clamp_min(0.0)
    C_DOC = xb_phys[:, ix_DOC:ix_DOC+1].clamp_min(0.0)
    kC    = xb_phys[:, ix_kC:ix_kC+1]
    kFe   = xb_phys[:, ix_kFe:ix_kFe+1]
    KFe   = xb_phys[:, ix_KFe:ix_KFe+1]
    FeM   = xb_phys[:, ix_Fe3:ix_Fe3+1].clamp_min(0.0)
    eps  = 1e-12
    KNO3 = torch.full_like(C_NO3, 1e-6)
    KDOC = torch.full_like(C_DOC, 1e-6)
    r_NO3 = kC  * (C_NO3 / (KNO3 + C_NO3 + eps)) * (C_DOC / (KDOC + C_DOC + eps))
    r_Fe  = kFe * (FeM  / (KFe  + FeM  + eps))   * (C_DOC / (KDOC + C_DOC + eps))
    R     = r_NO3 + r_Fe
    residual = phi * dCdt + v * dCdx - (phi * DL) * d2Cdx2 - R
    return residual.abs().mean()

def pde_residual_abs_vector(model, X_scaled, mu_X_t, std_X_t, mu_y_t, std_y_t, device):
    X = torch.tensor(X_scaled, dtype=torch.float32, device=device, requires_grad=True)
    C_scaled = model(X)
    if not X.requires_grad:
        X.requires_grad_(True)
    C = C_scaled * std_y_t + mu_y_t
    std_x = std_X_t[ix_x].view(1, 1)
    std_t = std_X_t[ix_t].view(1, 1)
    ones = torch.ones_like(C)
    grad_C = torch.autograd.grad(C, X, grad_outputs=ones,
                                 create_graph=True, retain_graph=True)[0]
    dCdx_s = grad_C[:, ix_x:ix_x+1]
    dCdt_s = grad_C[:, ix_t:ix_t+1]
    dCdx = dCdx_s / std_x
    dCdt = dCdt_s / std_t
    grad_dCdx_s = torch.autograd.grad(dCdx_s, X,
                                      grad_outputs=torch.ones_like(dCdx_s),
                                      create_graph=True, retain_graph=True)[0]
    d2Cdx2_s = grad_dCdx_s[:, ix_x:ix_x+1]
    d2Cdx2   = d2Cdx2_s / (std_x * std_x)
    xb_phys = X * std_X_t + mu_X_t
    phi     = xb_phys[:, ix_phi:ix_phi+1]
    v       = xb_phys[:, ix_v:ix_v+1]
    alphaL  = xb_phys[:, ix_alphaL:ix_alphaL+1]
    DL      = alphaL * v
    C_NO3 = (C).clamp_min(0.0)
    C_DOC = xb_phys[:, ix_DOC:ix_DOC+1].clamp_min(0.0)
    kC    = xb_phys[:, ix_kC:ix_kC+1]
    kFe   = xb_phys[:, ix_kFe:ix_kFe+1]
    KFe   = xb_phys[:, ix_KFe:ix_KFe+1]
    FeM   = xb_phys[:, ix_Fe3:ix_Fe3+1].clamp_min(0.0)
    eps  = 1e-12
    KNO3 = torch.full_like(C_NO3, 1e-6)
    KDOC = torch.full_like(C_DOC, 1e-6)
    r_NO3 = kC  * (C_NO3 / (KNO3 + C_NO3 + eps)) * (C_DOC / (KDOC + C_DOC + eps))
    r_Fe  = kFe * (FeM  / (KFe  + FeM  + eps))   * (C_DOC / (KDOC + C_DOC + eps))
    R     = r_NO3 + r_Fe
    residual = phi * dCdt + v * dCdx - (phi * DL) * d2Cdx2 - R
    return residual.abs().detach().cpu().numpy().reshape(-1)

def bc_losses(model, X_inlet_scaled, X_outlet_scaled, Cref_inlet_phys, mu_X_t, std_X_t, mu_y_t, std_y_t):
    pred_in_scaled = model(X_inlet_scaled)
    C_in_phys = pred_in_scaled * std_y_t + mu_y_t
    loss_inlet = torch.mean((C_in_phys - Cref_inlet_phys) ** 2)
    pred_out_scaled = model(X_outlet_scaled)
    C_out_phys = pred_out_scaled * std_y_t + mu_y_t
    ones = torch.ones_like(C_out_phys)
    grad_C = torch.autograd.grad(C_out_phys, X_outlet_scaled, grad_outputs=ones,
                                 create_graph=True, retain_graph=True)[0]
    dCdx_s = grad_C[:, ix_x:ix_x+1]
    std_x  = std_X_t[ix_x].view(1,1)
    dCdx_phys = dCdx_s / std_x
    loss_outlet = torch.mean(dCdx_phys ** 2)
    return loss_inlet, loss_outlet


def load_df(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    df = df[df.iloc[:, 5] != -99].copy()
    const_cols = [c for c in CANDIDATE_CONST_COLS if c in df.columns]
    if not const_cols:
        raise ValueError("No invariant candidate columns found.")
    const_block = df[const_cols].copy()
    for c in const_cols:
        if pd.api.types.is_numeric_dtype(const_block[c]):
            const_block[c] = const_block[c].round(8)
    sim_key = const_block.astype(str).agg('|'.join, axis=1)
    sim_id, _ = pd.factorize(sim_key, sort=True)
    df['__sim_id__'] = sim_id
    return df

def split_by_sim(df, seed=42) -> Tuple[pd.DataFrame, pd.DataFrame, set, set]:
    n_sims = df['__sim_id__'].nunique()
    rng = np.random.default_rng(seed)
    all_sims = np.array(sorted(df['__sim_id__'].unique()))
    rng.shuffle(all_sims)
    if n_sims >= 200:
        n_train_sims, n_test_sims = 160, 40
    elif n_sims >= 2:
        n_test_sims  = max(1, int(round(0.2 * n_sims)))
        n_train_sims = n_sims - n_test_sims
    else:
        raise ValueError("Fewer than 2 sims.")
    train_sims = set(all_sims[:n_train_sims])
    test_sims  = set(all_sims[n_train_sims:n_train_sims + n_test_sims])
    return (df[df['__sim_id__'].isin(train_sims)].copy(),
            df[df['__sim_id__'].isin(test_sims)].copy(),
            train_sims, test_sims)

def add_inlet_cref(df, df_train, df_test):
    x_min_per_sim_all = df.groupby('__sim_id__')['dist_x'].transform('min')
    is_inlet_all = (df['dist_x'].values == x_min_per_sim_all.values)
    ref_map = (
        df.loc[is_inlet_all, ['__sim_id__', 'time', 'C_NO3']]
          .rename(columns={'C_NO3': 'Cref'})
          .copy()
    )
    df_train = df_train.merge(ref_map, on=['__sim_id__', 'time'], how='left')
    df_test  = df_test.merge(ref_map,  on=['__sim_id__', 'time'], how='left')
    return df_train, df_test

def fit_scalers(df_train, df_test):
    X_train_raw = df_train[input_columns].to_numpy(dtype=float)
    y_train_raw = df_train[output_column].to_numpy(dtype=float).reshape(-1,1)
    X_test_raw  = df_test[input_columns].to_numpy(dtype=float)
    y_test_raw  = df_test[output_column].to_numpy(dtype=float).reshape(-1,1)
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_train = scaler_X.fit_transform(X_train_raw)
    y_train = scaler_y.fit_transform(y_train_raw)
    X_test  = scaler_X.transform(X_test_raw)
    y_test  = scaler_y.transform(y_test_raw)
    return X_train, y_train, X_test, y_test, scaler_X, scaler_y


def train_one_member(seed:int,
                     df_train_in: pd.DataFrame,
                     scaler_X: StandardScaler,
                     scaler_y: StandardScaler,
                     save_path:str,
                     device) -> None:

    torch.manual_seed(seed)
    np.random.seed(seed)

    X_train = scaler_X.transform(df_train_in[input_columns].to_numpy(dtype=float))
    y_train = scaler_y.transform(df_train_in[output_column].to_numpy(dtype=float).reshape(-1,1))

    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)

    mu_X_t  = torch.tensor(scaler_X.mean_,  dtype=torch.float32, device=device)
    std_X_t = torch.tensor(scaler_X.scale_, dtype=torch.float32, device=device)
    mu_y_t  = torch.tensor(scaler_y.mean_,  dtype=torch.float32, device=device)
    std_y_t = torch.tensor(scaler_y.scale_, dtype=torch.float32, device=device)

    x_min_per_sim_tr = df_train_in.groupby('__sim_id__')['dist_x'].transform('min').values
    x_max_per_sim_tr = df_train_in.groupby('__sim_id__')['dist_x'].transform('max').values
    is_left_tr  = (df_train_in['dist_x'].values == x_min_per_sim_tr)
    is_right_tr = (df_train_in['dist_x'].values == x_max_per_sim_tr)
    left_pos  = np.where(is_left_tr)[0]
    right_pos = np.where(is_right_tr)[0]

    X_inlet_t  = torch.tensor(X_train[left_pos],  dtype=torch.float32, device=device)
    X_outlet_t = torch.tensor(X_train[right_pos], dtype=torch.float32, device=device).requires_grad_(True)
    Cref_all   = df_train_in['Cref'].to_numpy(dtype=float).reshape(-1,1)
    Cref_inlet_t = torch.tensor(Cref_all[left_pos], dtype=torch.float32, device=device)

    model = SurrogatePINN(input_dim=len(input_columns)).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR_ADAM, weight_decay=WD_ADAMW)

    def make_rg(x: torch.Tensor) -> torch.Tensor:
        return x.detach().clone().requires_grad_(True)

    model.train()
    num_train = X_train_t.shape[0]
    steps_per_epoch = max(1, (num_train + BATCH_SIZE - 1) // BATCH_SIZE)

    for epoch in range(EPOCHS_ADAM):
        perm = torch.randperm(num_train, device=device)
        ramp = min(epoch / max(1, warmup_epochs), 1.0)
        lam_phys  = lambda_phys_base  + (lambda_phys_max  - lambda_phys_base)  * ramp
        lam_redox = lambda_redox_base + (lambda_redox_max - lambda_redox_base) * ramp

        for b in range(steps_per_epoch):
            idx = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            xb = make_rg(X_train_t[idx])
            yb = y_train_t[idx]
            optimizer.zero_grad()

            preds_scaled = model(xb)
            data_loss = nn.MSELoss()(preds_scaled, yb)

            pde_loss_sup = pde_residual_mean_abs(
                C_scaled=preds_scaled, xb_scaled=xb,
                mu_X_t=mu_X_t, std_X_t=std_X_t,
                mu_y_t=mu_y_t, std_y_t=std_y_t,
                device=device
            )

            if USE_COLLOCATION:
                colloc_idx = torch.randint(0, num_train, (idx.shape[0] * 2,), device=device)
                x_colloc = make_rg(X_train_t[colloc_idx])
                preds_colloc = model(x_colloc)
                pde_loss_colloc = pde_residual_mean_abs(
                    C_scaled=preds_colloc, xb_scaled=x_colloc,
                    mu_X_t=mu_X_t, std_X_t=std_X_t,
                    mu_y_t=mu_y_t, std_y_t=std_y_t,
                    device=device
                )
            else:
                pde_loss_colloc = torch.tensor(0.0, device=device)

            preds_phys_np = (preds_scaled.detach().cpu().numpy() * scaler_y.scale_ + scaler_y.mean_).reshape(-1)
            df_batch = df_train_in.iloc[idx.cpu().numpy()].copy()
            df_batch['N(5)(mol/kgw)_predicted'] = np.clip(preds_phys_np, 0, None)

            nonneg_loss = torch.tensor(compute_nonnegativity_penalty(preds_phys_np), dtype=torch.float32, device=device)
            redox_loss  = torch.tensor(compute_redox_sequence_penalty(df_batch), dtype=torch.float32, device=device)

            pde_total = pde_loss_sup + COLLOC_MULTIPLIER * pde_loss_colloc
            loss = data_loss + lam_phys * pde_total + lambda_nonneg * nonneg_loss + lam_redox * redox_loss

            loss.backward()
            optimizer.step()

        X_outlet_t.requires_grad_(True)
        bc_in, bc_out = bc_losses(model, X_inlet_t, X_outlet_t, Cref_inlet_t, mu_X_t, std_X_t, mu_y_t, std_y_t)
        optimizer.zero_grad(set_to_none=True)
        (lambda_bc_in * bc_in + lambda_bc_out * bc_out).backward()
        optimizer.step()

        if epoch % 100 == 0 or epoch == EPOCHS_ADAM - 1:
            print(f"[Seed {seed}] Epoch {epoch:4d} | Data={data_loss.item():.5e} "
                  f"| PDEsup={pde_loss_sup.item():.5e} PDEcol={pde_loss_colloc.item():.5e} "
                  f"| BC_in={bc_in.item():.5e} BC_out={bc_out.item():.5e} "
                  f"| λ_phys={lam_phys:.2f} λ_redox={lam_redox:.2f}")

    X_full_all = X_train_t
    y_full_all = y_train_t

    optimizer_lbfgs = optim.LBFGS(
        model.parameters(),
        max_iter=LBFGS_MAX_ITER,
        history_size=50,
        line_search_fn="strong_wolfe"
    )

    def closure():
        optimizer_lbfgs.zero_grad()
        n_sup = min(65536, X_full_all.shape[0])
        idx_sup = torch.randint(0, X_full_all.shape[0], (n_sup,), device=device)
        Xb = X_full_all[idx_sup].detach().clone().requires_grad_(True)
        yb = y_full_all[idx_sup]
        preds_scaled_fb = model(Xb)
        data_loss_fb = nn.MSELoss()(preds_scaled_fb, yb)
        pde_fb = pde_residual_mean_abs(
            C_scaled=preds_scaled_fb, xb_scaled=Xb,
            mu_X_t=mu_X_t, std_X_t=std_X_t,
            mu_y_t=mu_y_t, std_y_t=std_y_t,
            device=device
        )
        n_bc_in  = min(8192, X_inlet_t.shape[0])
        n_bc_out = min(8192, X_outlet_t.shape[0])
        sel_in  = torch.randint(0, X_inlet_t.shape[0],  (n_bc_in,),  device=device)
        sel_out = torch.randint(0, X_outlet_t.shape[0], (n_bc_out,), device=device)
        X_inlet_sub  = X_inlet_t[sel_in]
        Cref_in_sub  = Cref_inlet_t[sel_in]
        X_outlet_sub = X_outlet_t[sel_out].detach().clone().requires_grad_(True)
        bc_in_fb, bc_out_fb = bc_losses(model, X_inlet_sub, X_outlet_sub, Cref_in_sub,
                                        mu_X_t, std_X_t, mu_y_t, std_y_t)
        total = (data_loss_fb
                 + lambda_phys_max * pde_fb
                 + lambda_bc_in * bc_in_fb
                 + lambda_bc_out * bc_out_fb)
        total.backward()
        return total

    print(f"\n[Seed {seed}] LBFGS fine-tuning...")
    optimizer_lbfgs.step(closure)

    torch.save(model.state_dict(), save_path)
    print(f"[Seed {seed}] Saved: {save_path}")


def predict_phys(model, X_scaled, scaler_y, device):
    with torch.no_grad():
        Xt = torch.tensor(X_scaled, dtype=torch.float32, device=device)
        y_scaled = model(Xt)
        y_phys = (y_scaled * float(scaler_y.scale_[0]) + float(scaler_y.mean_[0])).squeeze(1)
        return y_phys.detach().cpu().numpy()

def _torch_load_safe(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ALPHA = ALPHAS[0]

   
    MASTER_SEED = 12345  

    torch.manual_seed(MASTER_SEED)
    np.random.seed(MASTER_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MASTER_SEED)

    df = load_df(DATA_PATH)
    print(f"Detected unique simulations: {df['__sim_id__'].nunique()}")

    df_train_full, df_test, train_sims, test_sims = split_by_sim(df, seed=MASTER_SEED)
    df_train_full, df_test = add_inlet_cref(df, df_train_full, df_test)

    print(f"Train simulations: {len(train_sims)} | Train rows: {len(df_train_full)}")
    print(f"Test  simulations: {len(test_sims)} | Test  rows: {len(df_test)}")

    rng = np.random.default_rng(MASTER_SEED)
    tr_sims = np.array(sorted(df_train_full['__sim_id__'].unique()))
    rng.shuffle(tr_sims)

    n_cal = max(1, int(round(CAL_FRAC * len(tr_sims))))
    cal_sims = set(tr_sims[:n_cal])
    tr_sims_final = set(tr_sims[n_cal:])

    df_cal      = df_train_full[df_train_full['__sim_id__'].isin(cal_sims)].copy()
    df_train_in = df_train_full[df_train_full['__sim_id__'].isin(tr_sims_final)].copy()

    X_train, y_train, X_test, y_test, scaler_X, scaler_y = fit_scalers(df_train_in, df_test)
    X_cal = scaler_X.transform(df_cal[input_columns].to_numpy(dtype=float))
    y_cal_phys = df_cal[output_column].to_numpy(dtype=float).reshape(-1)

   
    rng_seeds = np.random.default_rng(MASTER_SEED + 1)  
    SEED_LIST = rng_seeds.choice(np.arange(0, 2**31 - 1, dtype=np.int64),
                            size=ENSEMBLE_SIZE, replace=False).tolist()


    model_paths: List[str] = []
    for i, seed in enumerate(SEED_LIST):
        if BOOTSTRAP_TRAIN_SIMS:
            rng_i = np.random.default_rng(seed)
            boots = rng_i.choice(np.array(sorted(tr_sims_final)),
                                 size=len(tr_sims_final), replace=True)
            df_tr_i = df_train_in[df_train_in['__sim_id__'].isin(boots)].copy()
        else:
            df_tr_i = df_train_in

        path_i = os.path.join(MODEL_DIR, f"surrogate_pinn_seed{i}.pt")
        train_one_member(int(seed), df_tr_i, scaler_X, scaler_y, path_i, device)
        model_paths.append(path_i)

    X_test_scaled = scaler_X.transform(df_test[input_columns].to_numpy(dtype=float))
    y_test_phys   = df_test[output_column].to_numpy(dtype=float).reshape(-1)

    preds_test_members = []
    preds_cal_members  = []
    for path in model_paths:
        m = SurrogatePINN(len(input_columns)).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        preds_test_members.append(predict_phys(m, X_test_scaled, scaler_y, device))
        preds_cal_members.append(predict_phys(m, X_cal,         scaler_y, device))

    preds_test_arr = np.stack(preds_test_members, axis=1)  
    preds_cal_arr  = np.stack(preds_cal_members,  axis=1) 

    mu_test = preds_test_arr.mean(axis=1)
    sd_test = preds_test_arr.std(axis=1, ddof=1)
    mu_cal  = preds_cal_arr.mean(axis=1)

    abs_err_cal = np.abs(mu_cal - y_cal_phys.reshape(-1))
    n_cal_pts = len(abs_err_cal)
    q_level = math.ceil((n_cal_pts + 1) * (1 - ALPHA)) / (n_cal_pts + 1)

    def _quantile_higher(a, q):
        try:
            return np.quantile(a, q, method="higher")         
        except TypeError:
            return np.quantile(a, q, interpolation="higher")   

    qhat = float(_quantile_higher(abs_err_cal, q_level))

    if BETA > 0.0:
        m0 = SurrogatePINN(len(input_columns)).to(device)
        m0.load_state_dict(torch.load(model_paths[0], map_location=device))
        m0.eval()
        mu_X_t  = torch.tensor(scaler_X.mean_,  dtype=torch.float32, device=device)
        std_X_t = torch.tensor(scaler_X.scale_, dtype=torch.float32, device=device)
        mu_y_t  = torch.tensor(scaler_y.mean_,  dtype=torch.float32, device=device)
        std_y_t = torch.tensor(scaler_y.scale_, dtype=torch.float32, device=device)
        r_vec = pde_residual_abs_vector(m0, X_test_scaled, mu_X_t, std_X_t, mu_y_t, std_y_t, device)
        r_norm = (r_vec - r_vec.min()) / (r_vec.ptp() + 1e-12)
        widen = 1.0 + BETA * r_norm
    else:
        widen = np.ones_like(mu_test)
        r_vec = np.zeros_like(mu_test)

    half_width = np.sqrt(qhat**2 + (Z_EPI * sd_test)**2) * widen
    lo = mu_test - half_width
    hi = mu_test + half_width

    rmse = np.sqrt(mean_squared_error(y_test_phys, mu_test))
    mae  = mean_absolute_error(y_test_phys, mu_test)
    r2   = r2_score(y_test_phys, mu_test)
    coverage = np.mean((y_test_phys >= lo) & (y_test_phys <= hi))
    width = np.mean(hi - lo)

    print("\n=== UQ Summary (Ensemble + Conformal) ===")
    print(f"Ensemble size: {len(model_paths)}   Seeds: {SEED_LIST[:len(model_paths)]}")
    print(f"Conformal alpha: {ALPHA:.3f}  target {(1-ALPHA)*100:.0f}%")
    print(f"qhat: {qhat:.6e}")
    print(f"RMSE: {rmse:.6e} | MAE: {mae:.6e} | R^2: {r2:.6f}")
    print(f"Coverage: {coverage*100:.2f}% | Mean PI width: {width:.6e}")

    out = df_test.copy().reset_index(drop=True)
    out['PINN_ens_mean'] = mu_test
    out['PINN_ens_sd_epi'] = sd_test
    out['conformal_qhat'] = qhat
    out['pde_residual_abs'] = r_vec
    out['widen_factor'] = widen
    out['PI_lo'] = lo
    out['PI_hi'] = hi
    out = out.sort_values(['time', 'dist_x'])
    out_fn = os.path.join(OUT_DIR, "Benchmark4_uq.csv")
    out.to_csv(out_fn, index=False)
    print(f"Saved UQ CSV: {out_fn}")

    manifest = {
        "DATA_PATH": DATA_PATH,
        "ENSEMBLE_SIZE": ENSEMBLE_SIZE,
        "MASTER_SEED": MASTER_SEED,
        "SEEDS_USED": SEED_LIST[:len(model_paths)],
        "BOOTSTRAP_TRAIN_SIMS": BOOTSTRAP_TRAIN_SIMS,
        "CAL_FRAC": CAL_FRAC,
        "ALPHA": ALPHA,
        "Z_EPI": Z_EPI,
        "BETA": BETA,
        "POINT_METRICS": {"RMSE": float(rmse), "MAE": float(mae), "R2": float(r2)},
        "COVERAGE": float(coverage),
        "MEAN_PI_WIDTH": float(width),
        "MODEL_PATHS": model_paths,
        "SCALER": {
            "mu_X": scaler_X.mean_.tolist(),
            "std_X": scaler_X.scale_.tolist(),
            "mu_y": float(scaler_y.mean_[0]),
            "std_y": float(scaler_y.scale_[0]),
        }
    }
    with open(os.path.join(OUT_DIR, "run_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Saved manifest: {os.path.join(OUT_DIR, 'run_manifest.json')}")

if __name__ == "__main__":
    main()